<a href="https://colab.research.google.com/github/alsedawy/AI/blob/main/X_AutoAlign_RAG.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [22]:
# =====================================================================
# خلية متكاملة ومستقلة 100%: Corrective RAG (CRAG) مع واجهة Gradio تفاعلية
# =====================================================================

# 1. تثبيت كافة المكتبات اللازمة أوتوماتيكياً
!pip install -q langchain-text-splitters sentence-transformers rank_bm25 faiss-cpu transformers torch gradio

import pandas as pd
import numpy as np
import faiss
import torch
import gradio as gr
from langchain_text_splitters import RecursiveCharacterTextSplitter
from sentence_transformers import SentenceTransformer
from rank_bm25 import BM25Okapi
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

# ---------------------------------------------------------------------
# 2. تحميل البيانات وتجهيز النصوص
# ---------------------------------------------------------------------
print("1/5. جاري تحميل قاعدة البيانات وتجهيز النصوص...")
url = "https://raw.githubusercontent.com/sahil280114/codealpaca/master/data/code_alpaca_20k.json"
df = pd.read_json(url)

documents = []
for idx, row in df.iterrows():
    doc_text = f"Instruction: {row['instruction']}\nSolution:\n{row['output']}"
    documents.append(doc_text)

# تقطيع الوثائق إلى Chunks
text_splitter = RecursiveCharacterTextSplitter(chunk_size=300, chunk_overlap=50, length_function=len)

chunks = []
for i, doc in enumerate(documents[:2000]):
    split_texts = text_splitter.split_text(doc)
    for j, chunk in enumerate(split_texts):
        chunks.append({"chunk_id": f"doc_{i}_chunk_{j}", "text": chunk})

corpus_texts = [c['text'] for c in chunks]

# ---------------------------------------------------------------------
# 3. بناء الفهارس (BM25 + FAISS Vector Store)
# ---------------------------------------------------------------------
print("2/5. بناء الفهارس الهجينة (BM25 + FAISS Vector Store)...")
tokenized_corpus = [text.lower().split() for text in corpus_texts]
bm25 = BM25Okapi(tokenized_corpus)

embedder = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')
embeddings = embedder.encode(corpus_texts, convert_to_numpy=True)

# L2 Normalization لضمان دقة Cosine Similarity
faiss.normalize_L2(embeddings)
dimension = embeddings.shape[1]
faiss_index = faiss.IndexFlatIP(dimension)
faiss_index.add(embeddings.astype('float32'))

# ---------------------------------------------------------------------
# 4. محرك البحث الهجين (Hybrid RRF Search)
# ---------------------------------------------------------------------
def hybrid_rrf_search(query, k=60, top_n=3):
    tokenized_query = query.lower().split()
    bm25_indices = np.argsort(bm25.get_scores(tokenized_query))[::-1][:20]

    query_vec = embedder.encode([query], convert_to_numpy=True).astype('float32')
    faiss.normalize_L2(query_vec)
    _, faiss_indices = faiss_index.search(query_vec, k=20)

    rrf_scores = {}
    for rank, idx in enumerate(bm25_indices):
        rrf_scores[idx] = rrf_scores.get(idx, 0) + 1.0 / (k + rank + 1)
    for rank, idx in enumerate(faiss_indices[0]):
        rrf_scores[idx] = rrf_scores.get(idx, 0) + 1.0 / (k + rank + 1)

    sorted_docs = sorted(rrf_scores.items(), key=lambda x: x[1], reverse=True)
    return [doc_idx for doc_idx, score in sorted_docs[:top_n]]

# ---------------------------------------------------------------------
# 5. وحدة تقييم CRAG المتقدمة (Corrective Evaluator)
# ---------------------------------------------------------------------
def crag_evaluator(query, retrieved_chunk_indices, upper_threshold=0.60, lower_threshold=0.48):
    query_vec = embedder.encode([query], convert_to_numpy=True)
    faiss.normalize_L2(query_vec)

    best_score = -1.0
    best_chunk_text = ""

    for idx in retrieved_chunk_indices:
        chunk_text = chunks[idx]['text']
        chunk_vec = embedder.encode([chunk_text], convert_to_numpy=True)
        faiss.normalize_L2(chunk_vec)

        sim = np.dot(query_vec, chunk_vec.T)[0][0]
        if sim > best_score:
            best_score = sim
            best_chunk_text = chunk_text

    # تصنيف CRAG الثلاثي
    if best_score >= upper_threshold:
        return "CORRECT", best_chunk_text, best_score
    elif best_score >= lower_threshold:
        return "AMBIGUOUS", best_chunk_text, best_score
    else:
        return "INCORRECT", None, best_score

# ---------------------------------------------------------------------
# 6. وحدة إعادة صياغة الاستعلام (Query Rewriter)
# ---------------------------------------------------------------------
def crag_query_rewriter(original_query):
    return f"Write python code for: {original_query}"

# ---------------------------------------------------------------------
# 7. تحميل نموذج التوليد (LLM Generator)
# ---------------------------------------------------------------------
print("3/5. جاري تحميل نموذج التوليد LLM...")
model_name = "google/flan-t5-base"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

def generate_rag_response(query, context_text):
    prompt = f"Extract and write only the Python code from the context that answers the question.\nContext: {context_text}\nCode:"
    inputs = tokenizer(prompt, return_tensors="pt", max_length=512, truncation=True)

    outputs = model.generate(
        **inputs,
        max_new_tokens=60,
        no_repeat_ngram_size=2,
        early_stopping=True,
        num_beams=3
    )
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

# ---------------------------------------------------------------------
# 8. دالة الـ Pipeline للربط مع الواجهة Gradio
# ---------------------------------------------------------------------
def process_gradio_query(user_query):
    top_indices = hybrid_rrf_search(user_query, top_n=3)
    status, context, score = crag_evaluator(user_query, top_indices)

    score_str = f"{score:.2f}"

    if status == "CORRECT":
        status_msg = f"✅ ACCEPTED ({status}) - تم قبول المستندات والتمرير المباشر للتوليد."
        response = generate_rag_response(user_query, context)
        context_display = context

    elif status == "AMBIGUOUS":
        new_query = crag_query_rewriter(user_query)
        new_top_indices = hybrid_rrf_search(new_query, top_n=3)
        new_status, new_context, new_score = crag_evaluator(new_query, new_top_indices)

        score_str = f"{new_score:.2f} (بعد إعادة الصياغة)"

        if new_status in ["CORRECT", "AMBIGUOUS"] and new_context:
            status_msg = f"⚠️ REWRITTEN ({status} ➔ {new_status}) - تمت إعادة صياغة الاستعلام واستعادت البيانات."
            response = generate_rag_response(new_query, new_context)
            context_display = new_context
        else:
            status_msg = f"❌ REJECTED - عدم مطابقة المستندات حتى بعد إعادة الصياغة."
            response = "عذراً، لم يتم العثور على وثائق تقنية مطابقة بعد إعادة الصياغة لتفادي الهلوسة."
            context_display = "لا يوجد سياق مقبول (No Valid Context)."

    else: # INCORRECT
        status_msg = f"❌ REJECTED ({status}) - تفعيل آلية الرفض (Fallback Mechanism) لمنع الهلوسة."
        response = "عذراً، لم يتم العثور على وثائق تقنية موثوقة مطابقة لطلبك في قاعدة البيانات المتاحة لتفادي التوليد الخاطئ."
        context_display = "تم حجب السياق غير المطابق لحماية التوليد من الهلوسة."

    return status_msg, score_str, context_display, response

print("4/5. اكتمل إعداد النظام بالكامل!")

# ---------------------------------------------------------------------
# 9. تصميم وبناء واجهة Gradio التفاعلية
# ---------------------------------------------------------------------
print("5/5. جاري تشغيل واجهة Gradio...")

with gr.Blocks(title="Corrective RAG Pipeline", theme=gr.themes.Soft()) as demo:
    gr.Markdown(
        """
        # 🛡️ Self-Correcting Cross-Lingual RAG System
        ### نظام استرجاع هجين عابر للغات مع وحدة تصحيح ذاتي وآلية رفض لمنع الهلوسة (CRAG Framework)
        """
    )

    with gr.Row():
        with gr.Column(scale=1):
            query_input = gr.Textbox(
                label="أدخل الاستعلام (بالعربية أو الإنجليزية)",
                placeholder="مثال: كيف يمكن إنشاء مصفوفة تحتوي على الأرقام الزوجية في بايثون؟",
                lines=2
            )
            submit_btn = gr.Button("تشغيل الـ Pipeline 🚀", variant="primary")

            gr.Examples(
                examples=[
                    ["كيف يمكن إنشاء مصفوفة تحتوي على الأرقام الزوجية في بايثون؟"],
                    ["Write a python function to check if a string is palindrome"],
                    ["ما هي عاصمة جمهورية مصر العربية وأهم معالمها؟"]
                ],
                inputs=query_input,
                label="أمثلة جاهزة للتجربة"
            )

        with gr.Column(scale=1):
            status_output = gr.Textbox(label="حالة التقييم (CRAG Evaluator Status)", interactive=False)
            score_output = gr.Textbox(label="درجة التشابه الدلالي (Semantic Similarity)", interactive=False)
            context_output = gr.Textbox(label="السياق المسترجع والمعتمد (Filtered Context)", lines=5, interactive=False)
            response_output = gr.Code(label="إجابة النموذج النهائية (Generated Code)", language="python")

    submit_btn.click(
        fn=process_gradio_query,
        inputs=[query_input],
        outputs=[status_output, score_output, context_output, response_output]
    )

# تشغيل الواجهة ورابط الـ Public
demo.launch(share=True, debug=False)

1/5. جاري تحميل قاعدة البيانات وتجهيز النصوص...
2/5. بناء الفهارس الهجينة (BM25 + FAISS Vector Store)...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

3/5. جاري تحميل نموذج التوليد LLM...


Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.
/tmp/ipykernel_762/3101616480.py:176: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.
  with gr.Blocks(title="Corrective RAG Pipeline", theme=gr.themes.Soft()) as demo:


4/5. اكتمل إعداد النظام بالكامل!
5/5. جاري تشغيل واجهة Gradio...
Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://bca4073630a37d2da2.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
